# 09 - Promotion Recommendations

This notebook translates the final cluster profiles, top products, and association rules into simple promotion candidates. It does not refit or change the segmentation model.

## Scope

- Use only final post-clustering outputs.
- Keep one recommendation row per final cluster.
- Keep recommendations tied to observable evidence.
- Save one flat output file: `outputs/cluster_promotion_recommendations.csv`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../src")

from promotions import create_cluster_promotion_recommendations

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_colwidth", 120)

## Load Inputs

All inputs are existing final outputs from previous phases. The final cluster assignments and model outputs are not modified.

In [ ]:
cluster_profile = pd.read_csv("../outputs/kmeans_cluster_profile_summary.csv")
spending_shares = pd.read_csv("../outputs/kmeans_cluster_spending_shares.csv")
basket_profile = pd.read_csv("../outputs/cluster_basket_profile.csv")
top_products = pd.read_csv("../outputs/cluster_top_products.csv")
association_rules = pd.read_csv("../outputs/cluster_association_rules.csv")

print("cluster_profile:", cluster_profile.shape)
print("spending_shares:", spending_shares.shape)
print("basket_profile:", basket_profile.shape)
print("top_products:", top_products.shape)
print("association_rules:", association_rules.shape)

## Validate Inputs

The promotion phase expects all five final clusters to be present in every supporting table.

In [ ]:
expected_clusters = {0, 1, 2, 3, 4}

for name, frame in {
    "cluster_profile": cluster_profile,
    "spending_shares": spending_shares,
    "basket_profile": basket_profile,
    "top_products": top_products,
    "association_rules": association_rules,
}.items():
    assert set(frame["cluster"].unique()) == expected_clusters, name

print("Promotion input validation passed.")

## Promotion Plan

The table below contains the business interpretation for each cluster. It stays in the notebook because these labels and offers are analytical decisions, not reusable code.

In [ ]:
promotion_plan = pd.DataFrame(
    {
        "cluster": [0, 1, 2, 3, 4],
        "persona_name": [
            "Promo-Sensitive Older Tech Snackers",
            "Mainstream Grocery Families",
            "Vegetarian-Leaning Family Households",
            "Demanding Personal-Care and Tech Shoppers",
            "High-Value Large Loyal Families",
        ],
        "persona_summary": [
            "Older, loyal customers with high promotion usage and visible technology and snack signals.",
            "The broad main segment, dominated by grocery spend and family-oriented fresh-food baskets.",
            "Family households with the strongest vegetable share and very low meat and fish shares.",
            "Lower-loyalty customers with the highest complaints, strong personal-care baskets, and the highest electronics share.",
            "The highest-spending and most tenured family segment, with very large households and breakfast/grocery baskets.",
        ],
        "recommended_promotion": [
            "Offer a tech-accessory bundle paired with energy drinks or snack bars, with a loyalty-card discount.",
            "Use simple grocery and fresh-produce multi-buy offers for repeat basket items.",
            "Create a healthy family basket around vegetables, cooking basics, baby items, and pet essentials.",
            "Use a retention-focused personal-care replenishment bundle with optional tech-accessory cross-sell.",
            "Offer premium loyalty rewards and family breakfast bundles rather than deep discounts.",
        ],
        "promotion_type": [
            "Loyalty discount bundle",
            "High-volume grocery offer",
            "Lifestyle and family basket",
            "Retention and replenishment bundle",
            "Premium loyalty family bundle",
        ],
        "products_to_promote": [
            "airpods, bluetooth headphones, energy drink, energy bar, protein bar",
            "asparagus, tomatoes, spinach, carrots, avocado, salad",
            "asparagus, cooking oil, cereals, milk, babies food, dog food, cat food",
            "toothpaste, deodorant, shampoo, shower gel, tooth brush, airpods",
            "cereals, fresh bread, butter, eggs, milk, honey, tea",
        ],
        "risk_or_caveat": [
            "Spend is lower than other mature segments, so keep discounts controlled.",
            "This is the largest and broadest cluster, so promotions should stay general.",
            "This is vegetarian-leaning, not strictly vegetarian, because some meat products still appear.",
            "Use a professional label in the report; the informal Karen-like idea is only an interpretation of complaints and product behaviour.",
            "High-value customers may not need heavy discounts; focus on retention and value.",
        ],
        "product_terms": [
            ["airpods", "bluetooth headphones", "energy drink", "energy bar", "protein bar", "laptop"],
            ["asparagus", "tomatoes", "spinach", "carrots", "avocado", "salad"],
            ["asparagus", "cooking oil", "cereals", "milk", "babies food", "dog food", "cat food", "napkins"],
            ["toothpaste", "deodorant", "shampoo", "shower gel", "tooth brush", "airpods"],
            ["cereals", "fresh bread", "butter", "eggs", "milk", "honey", "tea", "bacon"],
        ],
        "rule_terms": [
            ["airpods", "bluetooth headphones", "energy drink"],
            ["salad", "avocado", "spinach", "carrots", "tomatoes"],
            ["cooking oil", "napkins", "babies food", "dog food"],
            ["airpods", "bluetooth headphones", "laptop", "energy drink"],
            ["cereals", "fresh bread", "butter", "eggs", "honey", "bacon"],
        ],
    }
)

promotion_plan[["cluster", "persona_name", "promotion_type"]]

## Create Recommendations

Each recommendation combines the business persona, supporting metrics, top basket products, and strongest association rules.

In [ ]:
promotion_recommendations = create_cluster_promotion_recommendations(
    promotion_plan,
    cluster_profile,
    spending_shares,
    basket_profile,
    top_products,
    association_rules,
)

promotion_recommendations.to_csv(
    "../outputs/cluster_promotion_recommendations.csv",
    index=False,
)

promotion_recommendations

## Persona and Promotion Summary

In [ ]:
promotion_recommendations[
    [
        "cluster",
        "persona_name",
        "promotion_type",
        "products_to_promote",
        "risk_or_caveat",
    ]
]

## Segment Value Context

Average lifetime spend helps decide whether a segment should receive deep discounts, broad grocery offers, or loyalty/retention incentives.

In [ ]:
plot_data = cluster_profile.merge(
    promotion_recommendations[["cluster", "persona_name"]],
    on="cluster",
    how="left",
)

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(
    data=plot_data,
    x="cluster",
    y="total_lifetime_spend_mean",
    ax=ax,
    color="#5DA5DA",
)
ax.set_title("Average Lifetime Spend by Final Cluster")
ax.set_xlabel("Cluster")
ax.set_ylabel("Average lifetime spend")
plt.tight_layout()

## Interpretation

- Cluster 0 can receive controlled loyalty discounts around tech accessories and snacks because it is promotion-sensitive but lower-spend.
- Cluster 1 should receive broad grocery and fresh-product offers because it is the largest, most general segment.
- Cluster 2 is the vegetarian-leaning family profile: vegetables are high and meat/fish shares are very low, but it should not be described as strictly vegetarian.
- Cluster 3 matches the expected demanding/Karen-like behaviour through complaints, low loyalty, personal-care products, and electronics, but the report should use a professional name.
- Cluster 4 is high-value and family-heavy, so premium loyalty bundles are safer than heavy discounting.

## Validation

In [ ]:
saved_recommendations = pd.read_csv("../outputs/cluster_promotion_recommendations.csv")

expected_columns = [
    "cluster",
    "persona_name",
    "persona_summary",
    "main_evidence",
    "recommended_promotion",
    "promotion_type",
    "products_to_promote",
    "supporting_products",
    "supporting_association_rules",
    "risk_or_caveat",
]

assert saved_recommendations.columns.tolist() == expected_columns
assert saved_recommendations.shape == (5, len(expected_columns))
assert set(saved_recommendations["cluster"]) == expected_clusters
assert saved_recommendations.notna().all().all()
assert Path("../outputs/cluster_promotion_recommendations.csv").exists()

print("Promotion recommendation output validation passed.")